# 01 · Bronze — Azure maliyet export'larını ham olarak al

**Ne yapar?** Azure Cost Management'ın her gün ADLS Gen2'ye yazdığı **FOCUS 1.0** Parquet dosyalarını
OneLake shortcut'ı (`Files/costs`) üzerinden okur ve **hiçbir değeri değiştirmeden** `bronze_costs`
Delta tablosuna yazar.

| Eklenen kolon | Neden |
|---|---|
| `_source_file` | Satırın hangi export dosyasından geldiği (soy ağacı / lineage) |
| `_file_modification_time` | Export çalışmasının zamanı — Silver'da *en güncel çalışmayı* seçmek için |
| `_ingestion_timestamp` | Bronze'a yüklenme zamanı |

> Export klasör yapısı: `costs/focus/<abonelik-id>/<export-adı>/<yyyyMMdd-yyyyMMdd>/<run-id>/part_*.parquet` (+ `manifest.json`)
>
> Birden fazla abonelik varsa her aboneliğin export'u kendi klasörüne yazar; hepsi bu notebook tarafından tek seferde okunur.

In [ ]:
import re

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

SOURCE_PATH = "Files/costs/"            # OneLake shortcut -> ADLS Gen2 container "costs"
TARGET_TABLE = "Tables/dbo/bronze_costs"
EXPECTED_COLUMNS = ["ChargePeriodStart", "EffectiveCost", "BilledCost", "ServiceName", "ResourceId", "BillingCurrency"]


def delta_safe_names(columns):
    """Delta column names cannot contain spaces or punctuation; keep the rest of the name intact."""
    seen, names = set(), []
    for original in columns:
        name = re.sub(r"_+", "_", re.sub(r"[^A-Za-z0-9_]", "_", original)) or "column"
        candidate, i = name, 2
        while candidate.lower() in seen:
            candidate, i = f"{name}_{i}", i + 1
        seen.add(candidate.lower())
        names.append(candidate)
    return names

## 1. Parquet dosyalarını oku

`recursiveFileLookup` tüm ay/run klasörlerini gezer, `pathGlobFilter` yalnızca `.parquet` dosyalarını alır
(`manifest.json` hariç), `mergeSchema` ise aylar arasında eklenen yeni FOCUS kolonlarını tolere eder.

In [ ]:
try:
    raw = (
        spark.read.format("parquet")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.parquet")
        .option("mergeSchema", "true")
        .load(SOURCE_PATH)
    )
except AnalysisException as exc:
    raise RuntimeError(
        "Files/costs altında Parquet dosyası yok. Export'u çalıştırın: scripts/02-run-cost-export.ps1"
    ) from exc

df_bronze = raw.select(
    *[F.col(f"`{src}`").alias(dst) for src, dst in zip(raw.columns, delta_safe_names(raw.columns))],
    F.col("_metadata.file_path").alias("_source_file"),
    F.col("_metadata.file_modification_time").alias("_file_modification_time"),
    F.current_timestamp().alias("_ingestion_timestamp"),
)

print(f"Satır: {df_bronze.count():,}   Kolon: {len(df_bronze.columns)}")
missing = [c for c in EXPECTED_COLUMNS if c not in df_bronze.columns]
print("Beklenen FOCUS kolonları:", "TAMAM" if not missing else f"EKSİK -> {missing}")

## 2. Hangi export çalışmaları var?

Her export (abonelik) ve fatura dönemi için kaç *run* olduğunu gösterir. Aynı ay için birden fazla run
görürseniz endişelenmeyin: Silver her export için yalnızca en günceli kullanır.

In [ ]:
display(
    df_bronze.groupBy(
        F.regexp_extract("_source_file", r"^(.*)/\d{8}-\d{8}/", 1).alias("export_path"),
        F.regexp_extract("_source_file", r"/(\d{8}-\d{8})/", 1).alias("billing_period"),
        F.regexp_extract("_source_file", r"/\d{8}-\d{8}/([^/]+)/", 1).alias("export_run"),
    )
    .agg(F.count("*").alias("rows"), F.max("_file_modification_time").alias("run_time"))
    .orderBy("export_path", "billing_period", "run_time")
)

## 3. Bronze Delta tablosunu yaz (tam yenileme)

In [ ]:
(
    df_bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TARGET_TABLE)
)
print(f"bronze_costs yazıldı: {spark.read.format('delta').load(TARGET_TABLE).count():,} satır")